# Quantum ML for pesticide residue classification - Colab quick start

Runs the whole pipeline on Google Colab (CPU runtime is enough). Upload or clone the project folder first,
then run the cells in order. Quantum circuits are simulated exactly (statevector), no quantum hardware is needed.

**Research demonstration only - not a substitute for certified laboratory residue testing.**

In [ ]:
# 1. Get the project (choose ONE): upload a zip of the project folder, or clone your repository
# from google.colab import files; files.upload()      # then: !unzip -q pesticide-residue-qml.zip
# !git clone <your-repository-url> pesticide-residue-qml
%cd pesticide-residue-qml

In [ ]:
# 2. Install pinned dependencies
!pip install -q -r requirements.txt

In [ ]:
# 3. Dataset: official USDA PDP 2022 database (public domain).
# Primary source: https://www.ams.usda.gov/sites/default/files/media/2022PDPDatabase.zip
# If that host is blocked, the identical file (same SHA-256) is mirrored on GitHub.
import os, hashlib, urllib.request
os.makedirs('data/raw', exist_ok=True)
dst = 'data/raw/2022PDPDatabase.zip'
if not os.path.exists(dst):
    for url in ['https://www.ams.usda.gov/sites/default/files/media/2022PDPDatabase.zip',
                'https://raw.githubusercontent.com/bdyapa9/Commodities-and-pesticides-data/HEAD/2022PDPDatabase.zip']:
        try:
            urllib.request.urlretrieve(url, dst); break
        except Exception as e:
            print('failed', url, e)
print(hashlib.sha256(open(dst,'rb').read()).hexdigest())
print('expected 9e599c741c6122d1ea7f6ec7ed0509ed5c2dbe34084cd5531b05303cafa433af')

In [ ]:
# 4. Build the labelled dataset, splits and reports
!python main.py data

In [ ]:
# 5. Classical baselines (tuned with Optuna). Takes ~10-30 min on a free CPU runtime.
!python main.py classical

In [ ]:
# 6. Quantum / hybrid optimisation (KTA screening + Optuna + VQC search)
!python main.py optimize

In [ ]:
# 7. Final evaluation, ablation study, learning curve, figures
!python main.py evaluate
!python main.py ablation
!python main.py scaling
!python main.py figures

In [ ]:
import pandas as pd
pd.read_csv('results/metrics/comparison_table.csv').round(3)

In [ ]:
# 8. Tests
!python -m pytest -q tests

In [ ]:
# 9. Streamlit demo (Colab: expose the port with a tunnel, e.g. localtunnel)
# !pip install -q streamlit
# !streamlit run app.py &>/dev/null &
# !npx localtunnel --port 8501